In [2]:
from jupyter_dash import JupyterDash

import dash_leaflet as dl
from dash import dcc, html, ctx
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# import Model portion of MVC architecture
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Decouple database config from class
username = "aacuser"
password = "<YourPasswordHere>"
host="localhost" 
port=27017
db_name="aac"
collection_name="animals"

# Connect to database via CRUD Module instantion 
shelter = AnimalShelter(
    username=username,
    password=password, 
    host=host,
    port=port,
    db=db_name,
    collection=collection_name
)
# Read initial records into Dataframe
df = pd.DataFrame.from_records(shelter.read({}))

# Guard against MongoDB ObjectID crashing DataTable 
if '_id' in df.columns: # guard clause
    # set inplace=True to return new dataframe without the dropped column(s)
    df.drop(columns=['_id'], inplace=True)

# Helper function to generate an explicit, self-contained dl.Map
def build_map(lat, lon, breed, name, marker_key='init'):
    # Remote Leaflet icon URLs to resolve Codio proxy asset 404s
    icon_config = {
        "iconUrl": "https://unpkg.com/leaflet@1.7.1/dist/images/marker-icon.png",
        "iconRetinaUrl": "https://unpkg.com/leaflet@1.7.1/dist/images/marker-icon-2x.png",
        "shadowUrl": "https://unpkg.com/leaflet@1.7.1/dist/images/marker-shadow.png",
        "iconSize": [25, 41],
        "iconAnchor": [12, 41],
        "popupAnchor": [1, -34],
        "shadowSize": [41, 41]
    }
    
    return dl.Map(
        id='map-component',
        style={'width': '1000px', 'height': '500px', 'margin': '0 auto'},
        center=[30.75, -97.48],  # Centered on Austin, TX
        zoom=10,
        children=[
            dl.TileLayer(
                id="base-layer-id",
                url="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
            ),
            dl.Marker(
                id=f'marker-{marker_key}',
                position=[float(lat), float(lon)],
                icon=icon_config,
                children=[
                    dl.Tooltip(str(breed)),
                    dl.Popup([
                        html.H4("Animal Name:"),
                        html.P(str(name))
                    ])
                ]
            )
        ]
    )
        
# Coordinates of row 0 for quick initial map view
# (Column 13: Lat, Column 14: Lon, Column 4: Breed, Column 9: Name)
init_lat = df.iloc[0]['location_lat'] if 'location_lat' in df.columns else df.iloc[0,13]
init_lon = df.iloc[0]['location_long'] if 'location_long' in df.columns else df.iloc[0,14]
init_breed = df.iloc[0]['breed'] if 'breed' in df.columns else df.iloc[0,4]
init_name = df.iloc[0]['name'] if 'name' in df.columns else df.iloc[0,9]

#########################
# Dashboard Layout / View
#########################
# Leaflet CSS required for tile layers, markers, and popups to render in  Jupyter
external_stylesheets = ['https://unpkg.com/leaflet@1.7.1/dist/leaflet.css']
# Initialize the JupyterDash app with external stylesheets
app = JupyterDash('GraziosoSalvare', external_stylesheets=external_stylesheets)

# Load Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
with open(image_filename, 'rb') as f:
    encoded_image = base64.b64encode(f.read()).decode('utf-8')

# Formatted columns list matching requirements
table_columns = [
    {"name": "Record #", "id": "rec_num", "deletable": False, "selectable": True},
    {"name": "Animal ID", "id": "animal_id", "deletable": False, "selectable": True},
    {"name": "Name", "id": "name", "deletable": False, "selectable": True},
    {"name": "Animal Type", "id": "animal_type", "deletable": False, "selectable": True},
    {"name": "Breed", "id": "breed", "deletable": False, "selectable": True},
    {"name": "Color", "id": "color", "deletable": False, "selectable": True},
    {"name": "Age", "id": "age_upon_outcome", "deletable": False, "selectable": True},
    {"name": "DOB", "id": "date_of_birth", "deletable": False, "selectable": True},
    {"name": "DateTime", "id": "datetime", "deletable": False, "selectable": True},
    {"name": "MonthYear", "id": "monthyear", "deletable": False, "selectable": True},
    {"name": "Outcome Type", "id": "outcome_type", "deletable": False, "selectable": True},
    {"name": "Outcome Subtype", "id": "outcome_subtype", "deletable": False, "selectable": True},
    {"name": "Age (weeks)", "id": "age_upon_outcome_in_weeks", "deletable": False, "selectable": True},
    {"name": "Sex", "id": "sex_upon_outcome", "deletable": False, "selectable": True},
    {"name": "Latitude", "id": "location_lat", "deletable": False, "selectable": True},
    {"name": "Longitude", "id": "location_long", "deletable": False, "selectable": True},
]

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    
    # Grazioso Salvare Logo with hyperlink to www.snhu.edu
    html.A(
        href='https://www.snhu.edu',
        target='_blank',
        children=[
            html.Img(
                src=f'data:image/png;base64,{encoded_image}',
                alt='Grazioso Salvare Logo',
                style={'height': '120px', 'display': 'block', 'margin': '10px auto'}
            )
        ]
    ),
    
    html.Center(html.B(html.H1('Grazioso Salvare Animal Shelter Dashboard'))), # Dashboard Header
    html.Hr(),
    
    # Interactive filtering options (Radio buttons)
    html.Div(
        className='radio-row',
        style={'display' : 'flex', 'gap': '10px', 'marginBottom': '15px'},
        children=[
            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label': 'Reset', 'value': 'RESET'},
                    {'label': 'Water Rescue', 'value': 'WATER'},
                    {'label': 'Mountain or Wilderness Rescue', 'value': 'MOUNTAIN'},
                    {'label': 'Disaster Rescue or Individual Tracking', 'value': 'DISASTER'}
                ],
                value='RESET', # Default selection on load
                inline=True, # aligned horizontally on page
                style={'display': 'flex', 'gap': '20px'}
            )
        ]
    ),
    html.Hr(),
    
    # Interactive Data Table
    dash_table.DataTable(
        id='datatable-id',
        data=df.to_dict('records'),
        columns=table_columns,
        style_cell={'textAlign': 'left', 'padding': '6px'},
        style_data={
            'color': 'black',
            'backgroundColor': 'white'
        },
        style_data_conditional=[
            {
                'if': {'row_index': 'odd'},
                'backgroundColor': 'rgb(220, 220, 220)'
            }
        ],
        style_header={
            'backgroundColor': 'rgb(210, 210, 210)', 
            'color': 'black',
            'fontWeight': 'bold'
        },
        editable=False,
        filter_action='native',
        filter_options={"placeholder_text": "Filter column..."},
        sort_action='native',
        sort_mode='multi',
        column_selectable=False,
        row_selectable='single',
        selected_columns=[],
        selected_rows=[0], # default set to row 0 so map load with initial marker
        page_action='native',
        page_current=0,
        page_size=10
    ),
    
    html.Br(),
    html.Hr(),

    # Side-by-side container for Pie Chart and Geolocation Map
    html.Div(
        className='row',
        style={'display' : 'flex', 'gap': '20px'},
        children=[
            html.Div(
                style={'flex': '1', 'minWidth': '450px'},
                children=[
                    dcc.Graph(
                        id='pie-chart-id',
                        figure=px.pie(title="Preferred Dog Breeds Breakdown"),
                        style={'height': '500px'}
                    )
                ]                  
            ),  
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'flex': '1', 'minWidth': '450px'},
                children=[
                    build_map(init_lat, init_lon, init_breed, init_name, '0')
                ]
            ),
        ]
    ),
    
    html.Br(),
    html.Hr(),

    # Student Idenfication Banner
    html.Div(
        "Student ID / Name: Grace Jungclas",
        style={
            'clear': 'both',
            'marginTop': '20px',
            'fontSize': '18px',
            'fontWeight': 'bold',
            'color': 'orange',
            'textAlign': 'center',
        }
    )    
])

#############################################
# Interaction Between Components / Controller
#############################################

# 1. Filter Animal Types on Radio Selection 
@app.callback(
    Output('datatable-id','data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_value):
    if filter_value == 'WATER':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_value == 'MOUNTAIN':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_value == 'DISASTER':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0}
        }
    else:
        # Default / Reset: retrieve unfiltered dataset
        query = {}
        
    records = shelter.read(query)
    dff = pd.DataFrame.from_records(records)
    
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
        
    return dff.to_dict('records')

# 2. Display Breed Distrobution in Pie Chart
@app.callback(
    Output('pie-chart-id', 'figure'),
    [Input('datatable-id', 'derived_virtual_data')] # using virtual (complete dataset after native filtering and sorting of rows)
)
def update_graphs(viewData):
    # fallback to global df if table data has not yet intialized
    if not viewData:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)
        
    # Return empty figure if dataframe is empty or breed column is missing
    if dff.empty or 'breed' not in dff.columns:
        empty_fig = px.pie(title="No data available")
        return empty_fig
    
    # Filter to Dogs if animal_type column is present
    if 'animal_type' in dff.columns:
        dogs_df = dff[dff['animal_type'] == 'Dog']
        if not dogs_df.empty:
            dff = dogs_df
    
    # Aggregate counts by breed
    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    
    # Cap to top 10 breeds for clean, rapid rednering
    if len(breed_counts) > 10:
        breed_counts = breed_counts.head(10)
    
    # Standard Plotly Express categorical pie chart    
    fig = px.pie(
        breed_counts,
        names='breed',
        values='count',
        title='Preferred Dog Breeds Breakdown'
    )
    
    # Formatting matching standard Plotly documentation
    fig.update_traces(textposition='inside', textinfo='percent+label')
    fig.update_layout(
        margin=dict(l=10, r=10, t=40, b=10)
    )
        
    return fig
    
# 3. Highlight Selected column Header (Preserving Zebra Stripes)
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # add zebra striping for visual clarity
    stripes=[{
        'if': {'row_index': 'odd'},
        'backgroundColor': 'rgb(220, 220, 220)'
    }]
    
    if not selected_columns:
        return stripes
    return stripes + [{
        'if': { 'column_id': i },
        'backgroundColor': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable

# 4. Update Geolocation Map on Row Selection
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_viewport_data"),
     Input('datatable-id', "derived_viewport_selected_rows")]
)
def update_map(view_data, selected_rows):
    # guard against empty query
    if not view_data:
        return [build_map(init_lat, init_lon, init_breed, init_name, 'fallback')]
    
    dff = pd.DataFrame.from_dict(view_data)
    
    # Single row selection (fallback to 0)
    row = 0 if not selected_rows else selected_rows[0]
    # Safety check: if row index exceeds current filtered data length
    if row >= len(dff):
        row = 0
        
    # Pull values by column names (or fallback index)
    # (Column 13: Lat, Column 14: Lon, Column 4: Breed, Column 9: Name)
    try:
        if 'location_lat' in dff.columns and 'location_long' in dff.columns:
            lat = float(dff.iloc[row]['location_lat'])
            lon = float(dff.iloc[row]['location_long'])
        else:
            lat = float(dff.iloc[row, 13])
            lon = float(dff.iloc[row, 14])
    except(ValueError, TypeError, IndexError):
        lat, lon = 30.75, -97.48 # Default coords for Austin, TX 
        
    breed = dff.iloc[row]['breed'] if 'breed' in dff.columns else dff.iloc[row, 4]
    name = dff.iloc[row]['name'] if 'name' in dff.columns else dff.iloc[row, 9] 
    
    # Returns the complete Map component into 'map-id'
    return [build_map(lat, lon, breed, name, str(row))]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()

Dash app running on https://appleamber-addressalpha-3000.codio.io/proxy/8050/
